# Fetch Sunnah Raw HTML

Notebook-first ingestion workflow for `https://sunnah.com/hisn`.

This notebook fetches:
- the main index page
- all per-dua pages `https://sunnah.com/hisn:{id}` for `1..267`

It stores raw HTML verbatim under `hisnul_muslim_sunnah/raw/` and writes a reproducible `manifest.json`.

In [6]:
from __future__ import annotations

import json
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

BASE_URL = "https://sunnah.com/hisn"
DUA_URL_TEMPLATE = "https://sunnah.com/hisn:{dua_id}"
DUA_ID_RANGE = range(1, 268)

NOTEBOOK_DIR = Path.cwd()
RAW_DIR = NOTEBOOK_DIR / "raw"
DUAS_DIR = RAW_DIR / "duas"
INDEX_PATH = RAW_DIR / "index.html"
MANIFEST_PATH = RAW_DIR / "manifest.json"
HADITH_PATH = RAW_DIR / "hadith.html"


REQUEST_HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; hisnul-muslim-sunnah-fetch/1.0; +https://sunnah.com/hisn)"
}
REQUEST_TIMEOUT = 45
MAX_ATTEMPTS = 3
RETRY_BACKOFF_SECONDS = 2.0
PER_REQUEST_DELAY_SECONDS = 0.5


In [7]:
def ensure_dirs() -> None:
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    DUAS_DIR.mkdir(parents=True, exist_ok=True)


def build_session() -> requests.Session:
    session = requests.Session()
    retry = Retry(
        total=MAX_ATTEMPTS,
        connect=MAX_ATTEMPTS,
        read=MAX_ATTEMPTS,
        status=MAX_ATTEMPTS,
        backoff_factor=RETRY_BACKOFF_SECONDS,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset(["GET"]),
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    session.headers.update(REQUEST_HEADERS)
    return session


SESSION = build_session()


def fetch_raw_html(url: str) -> tuple[str | None, int | None, str | None]:
    last_error = None
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            response = SESSION.get(url, timeout=REQUEST_TIMEOUT)
            response.raise_for_status()
            response.encoding = response.encoding or "utf-8"
            return response.text, response.status_code, None
        except requests.RequestException as exc:
            print(f"Error on attempt {attempt} for URL {url}: {exc}")
            last_error = f"{type(exc).__name__}: {exc}"

        curl_cmd = [
            "curl",
            "--http1.1",
            "--retry",
            str(MAX_ATTEMPTS),
            "--retry-delay",
            str(int(RETRY_BACKOFF_SECONDS)),
            "--retry-all-errors",
            "-A",
            REQUEST_HEADERS["User-Agent"],
            "-fsSL",
            url,
        ]
        curl_result = subprocess.run(curl_cmd, capture_output=True, check=False)
        if curl_result.returncode == 0:
            return curl_result.stdout.decode("utf-8", errors="replace"), 200, None

        curl_error = curl_result.stderr.decode("utf-8", errors="replace").strip()
        print(f"curl fallback failed on attempt {attempt} for URL {url}: {curl_error}")
        last_error = curl_error or last_error
        if attempt < MAX_ATTEMPTS:
            time.sleep(RETRY_BACKOFF_SECONDS * attempt)
        else:
            return None, None, last_error
    return None, None, last_error


def write_text(path: Path, text: str) -> None:
    path.write_text(text, encoding="utf-8")


def save_manifest(manifest: dict[str, Any]) -> None:
    MANIFEST_PATH.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


In [8]:
ensure_dirs()

manifest: dict[str, Any] = {
    "source": "sunnah.com/hisn",
    "fetched_at_utc": datetime.now(timezone.utc).isoformat(),
    "index": {},
    "duas": [],
    "dua_count": 267,
}

index_html, index_status, index_error = fetch_raw_html(BASE_URL)
index_entry: dict[str, Any] = {
    "url": BASE_URL,
    "path": str(INDEX_PATH.relative_to(NOTEBOOK_DIR)),
    "status_code": index_status,
}
if index_html is not None:
    print(f"Fetched index page successfully with status code {index_status}")
    write_text(INDEX_PATH, index_html)
else:
    print(f"Failed to fetch index page: {index_error}")
    index_entry["error"] = index_error
manifest["index"] = index_entry
save_manifest(manifest)

print(f"Saved index to: {INDEX_PATH}")


Fetched index page successfully with status code 200
Saved index to: /Users/rumman/work/quran_ar_en_word_scrapping/hisnul_muslim_sunnah/raw/index.html


In [9]:
# for dua_id in DUA_ID_RANGE:
#     url = DUA_URL_TEMPLATE.format(dua_id=dua_id)
#     rel_path = Path("raw") / "duas" / f"{dua_id:03d}.html"
#     out_path = NOTEBOOK_DIR / rel_path
#     html, status_code, error = fetch_raw_html(url)
#     entry: dict[str, Any] = {
#         "dua_id": dua_id,
#         "url": url,
#         "path": str(rel_path),
#         "status_code": status_code,
#     }
#     if html is not None:
#         write_text(out_path, html)
#     else:
#         entry["error"] = error
#     manifest["duas"].append(entry)
#     save_manifest(manifest)
#     time.sleep(PER_REQUEST_DELAY_SECONDS)

# save_manifest(manifest)
# print(f"Saved dua HTML under: {DUAS_DIR}")
# print(f"Saved manifest to: {MANIFEST_PATH}")


In [10]:
# manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
# successes = sum(1 for entry in manifest["duas"] if "error" not in entry)
# failures = [entry for entry in manifest["duas"] if "error" in entry]
# print("Index status:", manifest["index"].get("status_code"), manifest["index"].get("error"))
# print("Dua success count:", successes)
# print("Dua failure count:", len(failures))
# if failures:
#     print("First failures:")
#     for entry in failures[:10]:
#         print(entry)


In [11]:
from html.parser import HTMLParser

class AllHadithExtractor(HTMLParser):
    def __init__(self) -> None:
        super().__init__(convert_charrefs=False)
        self.capture = False
        self.depth = 0
        self.parts: list[str] = []

    def handle_starttag(self, tag: str, attrs) -> None:
        attrs_dict = dict(attrs)
        classes = set((attrs_dict.get("class") or "").split())
        if not self.capture and tag == "div" and "AllHadith" in classes:
            self.capture = True
            self.depth = 1
            self.parts.append(self.get_starttag_text())
            return
        if self.capture:
            if tag == "div":
                self.depth += 1
            self.parts.append(self.get_starttag_text())

    def handle_endtag(self, tag: str) -> None:
        if not self.capture:
            return
        self.parts.append(f"</{tag}>")
        if tag == "div":
            self.depth -= 1
            if self.depth == 0:
                self.capture = False

    def handle_startendtag(self, tag: str, attrs) -> None:
        if self.capture:
            self.parts.append(self.get_starttag_text())

    def handle_data(self, data: str) -> None:
        if self.capture:
            self.parts.append(data)

    def handle_comment(self, data: str) -> None:
        if self.capture:
            self.parts.append(f"<!--{data}-->")

    def handle_entityref(self, name: str) -> None:
        if self.capture:
            self.parts.append(f"&{name};")

    def handle_charref(self, name: str) -> None:
        if self.capture:
            self.parts.append(f"&#{name};")

    def handle_decl(self, decl: str) -> None:
        if self.capture:
            self.parts.append(f"<!{decl}>")

    def handle_pi(self, data: str) -> None:
        if self.capture:
            self.parts.append(f"<?{data}>")


index_html = INDEX_PATH.read_text(encoding="utf-8")
extractor = AllHadithExtractor()
extractor.feed(index_html)
hadith_html = "".join(extractor.parts).strip()

if not hadith_html:
    raise RuntimeError("Could not extract <div class=\"AllHadith\"> from raw/index.html")

HADITH_PATH.write_text(hadith_html + "\n", encoding="utf-8")
print(f"Saved core hadith block to: {HADITH_PATH}")
print(f"Extracted HTML size: {HADITH_PATH.stat().st_size} bytes")


Saved core hadith block to: /Users/rumman/work/quran_ar_en_word_scrapping/hisnul_muslim_sunnah/raw/hadith.html
Extracted HTML size: 637728 bytes


In [15]:
import re
CHAPTER_HTML_DIR = DUAS_DIR / "html"
CHAPTER_MANIFEST_PATH = CHAPTER_HTML_DIR / "manifest.json"
CHAPTER_ANCHOR_RE = re.compile(r'<a name=C(\d+)\.00></a>')
DUAS_REF_RE = re.compile(r'Hisn al-Muslim\s+(\d+)')

CHAPTER_HTML_DIR.mkdir(parents=True, exist_ok=True)

hadith_html = HADITH_PATH.read_text(encoding="utf-8")
anchor_matches = list(CHAPTER_ANCHOR_RE.finditer(hadith_html))
if len(anchor_matches) != 132:
    raise RuntimeError(f"Expected 132 chapter anchors, found {len(anchor_matches)}")

all_hadith_close = hadith_html.rfind("</div>")
if all_hadith_close == -1:
    raise RuntimeError("Could not find closing </div> for AllHadith block")

chapter_entries = []
for idx, match in enumerate(anchor_matches):
    chapter_number = int(match.group(1))
    start = match.start()
    if idx + 1 < len(anchor_matches):
        end = anchor_matches[idx + 1].start()
    else:
        end = all_hadith_close

    chapter_html = hadith_html[start:end].rstrip() + "\n"
    out_path = CHAPTER_HTML_DIR / f"chapter_{chapter_number:03d}.html"
    out_path.write_text(chapter_html, encoding="utf-8")

    dua_ids = sorted({int(x) for x in DUAS_REF_RE.findall(chapter_html)})
    chapter_entries.append({
        "chapter_number": chapter_number,
        "anchor": f"C{chapter_number}.00",
        "path": str(out_path.relative_to(NOTEBOOK_DIR)),
        "hadith_count": len(dua_ids),
        "first_dua_id": dua_ids[0] if dua_ids else None,
        "last_dua_id": dua_ids[-1] if dua_ids else None,
    })

chapter_manifest = {
    "chapter_count": len(chapter_entries),
    "source": "sunnah.com/hisn",
    "source_path": str(HADITH_PATH.relative_to(NOTEBOOK_DIR)),
    "chapters": chapter_entries,
}
CHAPTER_MANIFEST_PATH.write_text(json.dumps(chapter_manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

print(f"Saved {len(chapter_entries)} chapter files to: {CHAPTER_HTML_DIR}")
print(f"Saved chapter manifest to: {CHAPTER_MANIFEST_PATH}")


Saved 132 chapter files to: /Users/rumman/work/quran_ar_en_word_scrapping/hisnul_muslim_sunnah/raw/duas/html
Saved chapter manifest to: /Users/rumman/work/quran_ar_en_word_scrapping/hisnul_muslim_sunnah/raw/duas/html/manifest.json


In [16]:
import html
import re

PARSED_DUA_DIR = DUAS_DIR
CHAPTERS_JSON_PATH = PARSED_DUA_DIR / "chapters.json"

CHAPTER_NUMBER_DISPLAY_RE = re.compile(r'<div class=echapno>\s*(.*?)\s*</div>', re.S)
ENGLISH_CHAPTER_RE = re.compile(r'<div class=englishchapter>\s*(.*?)\s*</div>', re.S)
ARABIC_CHAPTER_NUMBER_DISPLAY_RE = re.compile(r'<div class=achapno>\s*(.*?)\s*</div>', re.S)
ARABIC_CHAPTER_RE = re.compile(r'<div class="arabicchapter arabic">\s*(.*?)\s*</div>', re.S)
DUA_CONTAINER_RE = re.compile(
    r'<div class="actualHadithContainer hadith_container_hisn"([^>]*)>(.*?)</div><!-- end actual hadith container -->',
    re.S,
)
ATTR_ID_RE = re.compile(r'\bid=(?:"([^"]+)"|([^\s>]+))')
DUA_ID_RE = re.compile(r'<div class="hadith_reference_sticky">Hisn al-Muslim\s+([0-9]+[a-z]?)</div>')
HADITH_NARRATED_RE = re.compile(r'<div class=hadith_narrated>\s*(.*?)\s*</div>', re.S)
TRANSLITERATION_RE = re.compile(r'<span class="transliteration">\s*(.*?)\s*</span>', re.S)
TRANSLATION_RE = re.compile(r'<span class="translation">\s*(.*?)\s*</span>', re.S)
ENGLISH_REFERENCE_RE = re.compile(r'<span class="hisn_english_reference">\s*(.*?)\s*</span>', re.S)
ARABIC_TEXT_RE = re.compile(r'<span class="arabic_text_details arabic">\s*(.*?)\s*</span>', re.S)
TAG_RE = re.compile(r'<[^>]+>')
WHITESPACE_RE = re.compile(r'\s+')
DIGIT_RE = re.compile(r'\d+')


def clean_html_text(value: str | None) -> str | None:
    if value is None:
        return None
    value = re.sub(r'<br\s*/?>', ' ', value)
    value = TAG_RE.sub(' ', value)
    value = html.unescape(value)
    value = WHITESPACE_RE.sub(' ', value).strip()
    return value or None


def require_match(pattern: re.Pattern[str], text: str, label: str, source_name: str):
    match = pattern.search(text)
    if match is None:
        raise RuntimeError(f"Could not find {label} in {source_name}")
    return match


def parse_display_number(value: str, label: str, source_name: str) -> int:
    match = DIGIT_RE.search(value)
    if match is None:
        raise RuntimeError(f"Could not parse {label} number in {source_name}: {value!r}")
    return int(match.group(0))


PARSED_DUA_DIR.mkdir(parents=True, exist_ok=True)

chapter_files = sorted(CHAPTER_HTML_DIR.glob('chapter_*.html'))
if len(chapter_files) != 132:
    raise RuntimeError(f"Expected 132 chapter HTML files, found {len(chapter_files)}")

chapters = []
all_dua_ids = []

for chapter_path in chapter_files:
    chapter_html = chapter_path.read_text(encoding='utf-8')
    source_name = chapter_path.name

    chapter_number_display = clean_html_text(
        require_match(CHAPTER_NUMBER_DISPLAY_RE, chapter_html, 'english chapter number', source_name).group(1)
    )
    english_title = clean_html_text(
        require_match(ENGLISH_CHAPTER_RE, chapter_html, 'english chapter title', source_name).group(1)
    )
    arabic_chapter_number_display = clean_html_text(
        require_match(ARABIC_CHAPTER_NUMBER_DISPLAY_RE, chapter_html, 'arabic chapter number', source_name).group(1)
    )
    arabic_title = clean_html_text(
        require_match(ARABIC_CHAPTER_RE, chapter_html, 'arabic chapter title', source_name).group(1)
    )

    if english_title and english_title.startswith('Chapter:'):
        english_title = english_title[len('Chapter:'):].strip()

    chapter_number = parse_display_number(chapter_number_display or '', 'chapter', source_name)

    duas = []
    for container_match in DUA_CONTAINER_RE.finditer(chapter_html):
        container_attrs = container_match.group(1)
        container_html = container_match.group(2)

        container_id_match = ATTR_ID_RE.search(container_attrs)
        container_id = None
        if container_id_match is not None:
            container_id = container_id_match.group(1) or container_id_match.group(2)

        dua_id_text = require_match(DUA_ID_RE, container_html, 'dua id', source_name).group(1)
        if not dua_id_text.isdigit():
            print(f"Skipping non-numeric Sunnah entry {dua_id_text} in {source_name}")
            continue
        dua_id = int(dua_id_text)
        all_dua_ids.append(dua_id)

        narration_match = HADITH_NARRATED_RE.search(container_html)
        transliteration_match = TRANSLITERATION_RE.search(container_html)
        translation_match = TRANSLATION_RE.search(container_html)
        english_reference_match = ENGLISH_REFERENCE_RE.search(container_html)
        arabic_match = ARABIC_TEXT_RE.search(container_html)

        dua_entry = {
            'dua_id': dua_id,
            'container_id': container_id,
            'english_narration': clean_html_text(narration_match.group(1) if narration_match else None),
            'transliteration': clean_html_text(transliteration_match.group(1) if transliteration_match else None),
            'translation': clean_html_text(translation_match.group(1) if translation_match else None),
            'english_reference': clean_html_text(english_reference_match.group(1) if english_reference_match else None),
            'arabic': clean_html_text(arabic_match.group(1) if arabic_match else None),
        }
        duas.append(dua_entry)

    chapters.append({
        'chapter_number': chapter_number,
        'chapter_number_display': chapter_number_display,
        'arabic_chapter_number_display': arabic_chapter_number_display,
        'english_title': english_title,
        'arabic_title': arabic_title,
        'source_file': chapter_path.name,
        'duas': duas,
    })

unique_dua_ids = sorted(set(all_dua_ids))
if len(chapters) != 132:
    raise RuntimeError(f"Expected 132 parsed chapters, found {len(chapters)}")
if len(all_dua_ids) != 267:
    raise RuntimeError(f"Expected 267 parsed duas, found {len(all_dua_ids)}")
if unique_dua_ids != list(range(1, 268)):
    raise RuntimeError('Parsed dua IDs are not a continuous 1..267 range')

payload = {
    'source': 'sunnah.com/hisn',
    'source_type': 'chapter_html_slices',
    'chapter_count': len(chapters),
    'dua_count': len(all_dua_ids),
    'chapters': chapters,
}

CHAPTERS_JSON_PATH.write_text(
    json.dumps(payload, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)

print(f"Saved parsed JSON to: {CHAPTERS_JSON_PATH}")
print(f"Chapters: {payload['chapter_count']}")
print(f"Duas: {payload['dua_count']}")
print(f"First chapter: {chapters[0]['chapter_number']} -> first dua {chapters[0]['duas'][0]['dua_id']}")
print(f"Last chapter: {chapters[-1]['chapter_number']} -> last dua {chapters[-1]['duas'][-1]['dua_id']}")


Skipping non-numeric Sunnah entry 75a in chapter_027.html
Saved parsed JSON to: /Users/rumman/work/quran_ar_en_word_scrapping/hisnul_muslim_sunnah/raw/duas/chapters.json
Chapters: 132
Duas: 267
First chapter: 1 -> first dua 1
Last chapter: 132 -> last dua 267
